# Mixed-traffic Fast-DeePC

Fixed benchmark: $T=1200$. This notebook runs the original hard-equality DeeP-LCC and C Fast-DeePC controllers for one braking disturbance, then writes one GIF.


## 1. Import packages and locate the repository


In [ ]:
from pathlib import Path
import ctypes as ct
import os
import shutil
import subprocess
import sys
import time

# Avoid severe BLAS oversubscription when CVXPY, SciPy, and the C DLL coexist.
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("OMP_NUM_THREADS", "1")

import numpy as np
import scipy.linalg as la
import cvxpy as cp
import matplotlib.pyplot as plt
import osqp
import scipy.sparse as sp

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
HERE = ROOT

print("Repository root:", ROOT)
print("CVXPY solvers:", cp.installed_solvers())


## 2. Build and load the C dense BoxQP solver


In [ ]:
CORE_SOURCE = ROOT / "src" / "deepc_boxqp_optimized.c"
WRAPPER_SOURCE = ROOT / "src" / "deepc_boxqp_optimized_ctypes_wrapper.c"
DLL = ROOT / "build" / "deep_lcc_kform_boxqp.dll"
DLL.parent.mkdir(exist_ok=True)
OPENBLAS_BIN = Path(r"C:\msys64\ucrt64\bin")
OPENBLAS_LIB = Path(r"C:\msys64\ucrt64\lib")

gcc_candidates = [
    shutil.which("gcc"),
    r"C:\msys64\ucrt64\bin\gcc.exe",
]
gcc = next((Path(p) for p in gcc_candidates if p and Path(p).exists()), None)
if gcc is None:
    raise FileNotFoundError("gcc was not found. Install MSYS2/UCRT64 gcc.")

source_mtime = max(CORE_SOURCE.stat().st_mtime, WRAPPER_SOURCE.stat().st_mtime)
needs_build = (not DLL.exists()) or DLL.stat().st_mtime < source_mtime
if needs_build:
    if (OPENBLAS_LIB / "libopenblas.dll.a").exists():
        command = [
            str(gcc), "-O3", "-shared", "-Wl,--export-all-symbols",
            "-o", str(DLL), str(CORE_SOURCE),
            f"-L{OPENBLAS_LIB}", "-lopenblas", "-lm",
        ]
    else:
        print("OpenBLAS was not found; using the slower standalone wrapper.")
        command = [
            str(gcc), "-O3", "-shared", "-Wl,--export-all-symbols",
            "-o", str(DLL), str(WRAPPER_SOURCE), "-lm",
        ]
    print("Compiling:", " ".join(command))
    subprocess.run(command, cwd=HERE, check=True)

if OPENBLAS_BIN.exists() and hasattr(os, "add_dll_directory"):
    _openblas_dll_directory = os.add_dll_directory(str(OPENBLAS_BIN))
lib = ct.CDLL(str(DLL))
BINT = ct.c_long

class BoxQPInfo(ct.Structure):
    _fields_ = [
        ("iterations", BINT),
        ("mu", ct.c_double),
        ("duality", ct.c_double),
        ("status", ct.c_int),
        ("chol_info", BINT),
    ]

class DeePCKForm(ct.Structure):
    _fields_ = [
        ("nfixed", BINT), ("nz", BINT), ("ncol", BINT), ("nrows", BINT),
        ("ridge", ct.c_double),
        ("A", ct.POINTER(ct.c_double)), ("Gchol", ct.POINTER(ct.c_double)),
        ("Hphys", ct.POINTER(ct.c_double)), ("Fphys", ct.POINTER(ct.c_double)),
        ("Hbox", ct.POINTER(ct.c_double)), ("Fbox", ct.POINTER(ct.c_double)),
        ("center", ct.POINTER(ct.c_double)), ("scale", ct.POINTER(ct.c_double)),
        ("hcenter", ct.POINTER(ct.c_double)), ("hbox", ct.POINTER(ct.c_double)),
        ("swarm", ct.POINTER(ct.c_double)), ("sout", ct.POINTER(ct.c_double)),
    ]

DOUBLE_PTR = ct.POINTER(ct.c_double)
lib.deepc_kform_offline.argtypes = [
    ct.POINTER(DeePCKForm), DOUBLE_PTR, DOUBLE_PTR, BINT, BINT, BINT,
    DOUBLE_PTR, DOUBLE_PTR, ct.c_double, DOUBLE_PTR, DOUBLE_PTR, DOUBLE_PTR,
]
lib.deepc_kform_offline.restype = ct.c_int
lib.deepc_kform_online_step.argtypes = [
    ct.POINTER(DeePCKForm), DOUBLE_PTR, DOUBLE_PTR, DOUBLE_PTR,
    ct.c_double, BINT, DOUBLE_PTR, ct.POINTER(BoxQPInfo),
]
lib.deepc_kform_online_step.restype = ct.c_int
lib.deepc_kform_recover_g.argtypes = [
    ct.POINTER(DeePCKForm), DOUBLE_PTR, DOUBLE_PTR, DOUBLE_PTR,
]
lib.deepc_kform_recover_g.restype = ct.c_int

def ptr(array):
    return np.asarray(array, dtype=np.float64).ctypes.data_as(DOUBLE_PTR)

def fmat(array):
    return np.asfortranarray(array, dtype=np.float64)

print("Loaded:", DLL)


## 3. Define the fixed platoon and Fast-DeePC parameters


In [ ]:
DT = 0.05
V_STAR = 15.0
S_STAR = 20.0
V_MAX = 30.0
S_ST = 5.0

ID = np.array([0, 0, 1, 0, 0, 1, 0, 0], dtype=int)
N_VEHICLES = len(ID)
POS_CAV = np.flatnonzero(ID == 1)
N_CAV = len(POS_CAV)
DIM_U, DIM_E, DIM_Y = N_CAV, 1, N_VEHICLES + N_CAV

T_INI = 20
HORIZON = 50
T_DATA = 1200                 # paper default; repository brake script uses 2000
CONTROL_TIME = 16.0          # covers brake, hold, recovery, and settling
CONTROL_STEPS = int(CONTROL_TIME / DT)
CONTROL_SHIFT = 1             # solve at every 0.05 s simulation step

WEIGHT_V = 1.0
WEIGHT_S = 0.5
WEIGHT_U = 0.1
LAMBDA_G = 10.0              # paper default; repository brake script uses 100
LAMBDA_Y = 1.0e4

# Fast soft formulation. These enforce the equalities approximately.
LAMBDA_CONSISTENCY = 1.0e4
RHO_FUTURE_OUTPUT = 1.0e4

U_MIN, U_MAX = -5.0, 2.0
SPACING_ERROR_MIN = 5.0 - S_STAR
SPACING_ERROR_MAX = 40.0 - S_STAR
VELOCITY_ERROR_MIN, VELOCITY_ERROR_MAX = -V_STAR, V_MAX - V_STAR

OFFLINE_NOISE = 0.0
ONLINE_NOISE = 0.0
SEED = 42

# Time-major output ordering: [v1,...,v8,s_CAV3,s_CAV6] at each step.
Q_STEP = np.r_[np.full(N_VEHICLES, WEIGHT_V), np.full(N_CAV, WEIGHT_S)]
Q_BAR = np.tile(Q_STEP, HORIZON)
R_BAR = np.full(HORIZON * DIM_U, WEIGHT_U)
SPACING_INDEX = np.concatenate([
    np.arange(k * DIM_Y + N_VEHICLES, (k + 1) * DIM_Y)
    for k in range(HORIZON)
])

print("CAV positions (1-based):", POS_CAV + 1)
print(f"g dimension at T={T_DATA}:", T_DATA - T_INI - HORIZON + 1)
print("Reduced BoxQP dimension:", HORIZON * (DIM_U + DIM_Y))


## 4. Define the nonlinear traffic plant and collect the fixed offline data


In [ ]:
class OVMVehicle:
    def __init__(self, alpha, beta, s_go, dt=DT):
        self.alpha = float(alpha)
        self.beta = float(beta)
        self.s_go = float(s_go)
        self.dt = float(dt)

    def optimal_velocity(self, spacing):
        if spacing <= S_ST:
            return 0.0
        if spacing >= self.s_go:
            return V_MAX
        phase = np.pi * (spacing - S_ST) / (self.s_go - S_ST)
        return 0.5 * V_MAX * (1.0 - np.cos(phase))

    def acceleration(self, velocity, spacing, lead_velocity, noise=0.0):
        desired = self.optimal_velocity(spacing)
        value = self.alpha * (desired - velocity)
        value += self.beta * (lead_velocity - velocity)
        value += float(noise)
        if spacing <= S_ST:
            value = U_MIN
        return float(np.clip(value, U_MIN, U_MAX))


# Table-I heterogeneous parameters, with nominal entries at CAV positions.
ALPHA = np.array([0.45, 0.75, 0.60, 0.70, 0.50, 0.60, 0.40, 0.80])
BETA  = np.array([0.60, 0.95, 0.90, 0.95, 0.75, 0.90, 0.80, 1.00])
S_GO  = np.array([38.0, 31.0, 35.0, 33.0, 37.0, 35.0, 39.0, 34.0])


class MixedTrafficPlatoon:
    def __init__(self, vehicle_id=ID):
        self.ID = np.asarray(vehicle_id, dtype=int)
        self.n = len(self.ID)
        self.pos_cav = np.flatnonzero(self.ID == 1)
        self.hdvs = [OVMVehicle(ALPHA[i], BETA[i], S_GO[i]) for i in range(self.n)]
        self.reset()

    def reset(self):
        self.v = np.full(self.n, V_STAR, dtype=np.float64)
        # Each heterogeneous HDV starts at its own OVM equilibrium; CAV spacing is 20 m.
        self.s = S_ST + (S_GO - S_ST) / np.pi * np.arccos(1.0 - 2.0 * V_STAR / V_MAX)
        self.s = self.s.astype(np.float64)
        self.s[self.pos_cav] = S_STAR

    def output(self):
        velocity_error = self.v - V_STAR
        spacing_error = self.s[self.pos_cav] - S_STAR
        return np.r_[velocity_error, spacing_error]

    def step(self, u_cav, v_head, acceleration_noise=None):
        u_cav = np.asarray(u_cav, dtype=np.float64).reshape(-1)
        if acceleration_noise is None:
            acceleration_noise = np.zeros(self.n)

        old_v = self.v.copy()
        old_s = self.s.copy()
        new_v = old_v.copy()
        new_s = old_s.copy()

        cav_lookup = {vehicle: j for j, vehicle in enumerate(self.pos_cav)}
        for i in range(self.n):
            lead_velocity = float(v_head if i == 0 else old_v[i - 1])
            if self.ID[i] == 1:
                acceleration = float(np.clip(u_cav[cav_lookup[i]], U_MIN, U_MAX))
            else:
                acceleration = self.hdvs[i].acceleration(
                    old_v[i], old_s[i], lead_velocity, acceleration_noise[i]
                )
            new_v[i] = np.clip(old_v[i] + DT * acceleration, 0.0, V_MAX)
            new_s[i] = old_s[i] + DT * (lead_velocity - old_v[i])

        self.v, self.s = new_v, new_s
        return self.output()


def brake_profile(number_of_steps):
    # Repository Experiment C: -5 m/s^2 for 2 s, hold 5 s, +2 m/s^2 for 5 s.
    velocity = np.empty(number_of_steps + HORIZON, dtype=np.float64)
    acceleration = np.zeros_like(velocity)
    velocity[0] = V_STAR
    for k in range(len(velocity) - 1):
        t = k * DT
        acceleration[k] = U_MIN if t < 2.0 else (0.0 if t < 7.0 else (U_MAX if t < 12.0 else 0.0))
        velocity[k + 1] = np.clip(velocity[k] + DT * acceleration[k], 0.0, V_STAR)
    return velocity, acceleration


def collect_offline_data(T, seed=SEED):
    rng = np.random.default_rng(seed)
    platoon = MixedTrafficPlatoon(ID)
    u_data = rng.uniform(-2.0, 2.0, size=(T, DIM_U))
    e_data = rng.uniform(-1.0, 1.0, size=(T, 1))
    noise = rng.uniform(-OFFLINE_NOISE, OFFLINE_NOISE, size=(T, N_VEHICLES))
    y_data = np.empty((T, DIM_Y), dtype=np.float64)
    for k in range(T):
        y_data[k] = platoon.step(u_data[k], V_STAR + e_data[k, 0], noise[k])
    return u_data, e_data, y_data


def block_hankel(data, depth):
    data = np.asarray(data, dtype=np.float64)
    if data.ndim == 1:
        data = data[:, None]
    T, dimension = data.shape
    columns = T - depth + 1
    if columns <= 0:
        raise ValueError(f"T={T} is shorter than Hankel depth {depth}")
    H = np.empty((depth * dimension, columns), dtype=np.float64)
    for j in range(columns):
        H[:, j] = data[j:j + depth].reshape(-1)
    return H


def build_hankel_blocks(u_data, e_data, y_data):
    depth = T_INI + HORIZON
    Hu = block_hankel(u_data, depth)
    He = block_hankel(e_data, depth)
    Hy = block_hankel(y_data, depth)
    return {
        "Up": Hu[:T_INI * DIM_U],
        "Uf": Hu[T_INI * DIM_U:],
        "Ep": He[:T_INI * DIM_E],
        "Ef": He[T_INI * DIM_E:],
        "Yp": Hy[:T_INI * DIM_Y],
        "Yf": Hy[T_INI * DIM_Y:],
    }


u_data, e_data, y_data = collect_offline_data(T_DATA)
BLOCKS = build_hankel_blocks(u_data, e_data, y_data)
print({name: matrix.shape for name, matrix in BLOCKS.items()})


## 5. Build the original and C Fast-DeePC controllers


In [ ]:
def _assign(parameter, value):
    parameter.value = np.asarray(value, dtype=np.float64).reshape(-1)


class OriginalDeepLCC:
    def __init__(self, blocks):
        self.B = blocks
        nc = blocks["Up"].shape[1]
        ny = HORIZON * DIM_Y
        self.g = cp.Variable(nc)
        self.uini = cp.Parameter(T_INI * DIM_U)
        self.eini = cp.Parameter(T_INI * DIM_E)
        self.yini = cp.Parameter(T_INI * DIM_Y)
        self.ef = cp.Parameter(HORIZON * DIM_E)
        self.r = cp.Parameter(ny)

        # Official-style g-only formulation. These expressions eliminate the
        # separate u, y, and sigma_y variables without changing the problem.
        self.u_expr = blocks["Uf"] @ self.g
        self.y_expr = blocks["Yf"] @ self.g
        self.sigma_expr = blocks["Yp"] @ self.g - self.yini

        objective = 0.5 * cp.sum(
            cp.multiply(Q_BAR, cp.square(self.y_expr - self.r))
        )
        objective += 0.5 * cp.sum(
            cp.multiply(R_BAR, cp.square(self.u_expr))
        )
        objective += LAMBDA_G * cp.norm1(self.g)
        objective += LAMBDA_Y * cp.sum_squares(self.sigma_expr)

        constraints = [
            blocks["Up"] @ self.g == self.uini,
            blocks["Ep"] @ self.g == self.eini,
            blocks["Ef"] @ self.g == self.ef,
            self.u_expr >= U_MIN,
            self.u_expr <= U_MAX,
            self.y_expr[SPACING_INDEX] >= SPACING_ERROR_MIN,
            self.y_expr[SPACING_INDEX] <= SPACING_ERROR_MAX,
        ]
        self.problem = cp.Problem(cp.Minimize(objective), constraints)
        if not self.problem.is_dpp():
            raise RuntimeError("Original g-only DeeP-LCC problem is not DPP")

    def solve(self, uini, eini, yini, ef, reference):
        _assign(self.uini, uini)
        _assign(self.eini, eini)
        _assign(self.yini, yini)
        _assign(self.ef, ef)
        _assign(self.r, reference)

        start = time.perf_counter()
        solver_used = "CLARABEL"
        try:
            self.problem.solve(
                solver=cp.CLARABEL,
                warm_start=True,
                tol_gap_abs=1e-5,
                tol_gap_rel=1e-5,
                tol_feas=1e-5,
                max_iter=500,
                verbose=False,
            )
        except cp.SolverError:
            solver_used = "OSQP"
            self.problem.solve(
                solver=cp.OSQP,
                warm_start=True,
                eps_abs=1e-4,
                eps_rel=1e-4,
                max_iter=10000,
                polishing=False,
                adaptive_rho=True,
                check_termination=25,
                verbose=False,
            )

        elapsed = time.perf_counter() - start
        if self.problem.status not in (cp.OPTIMAL, cp.OPTIMAL_INACCURATE):
            raise RuntimeError(
                f"Original DeeP-LCC status: {self.problem.status} "
                f"with {solver_used}"
            )

        return {
            "u": np.asarray(self.u_expr.value).reshape(HORIZON, DIM_U),
            "y": np.asarray(self.y_expr.value).reshape(HORIZON, DIM_Y),
            "objective": float(self.problem.value),
            "time": elapsed,
            "iterations": int(self.problem.solver_stats.num_iters or 0),
            "solver": solver_used,
        }

class SoftDeepLCC:
    def __init__(self, blocks):
        self.B = blocks
        nc = blocks["Up"].shape[1]
        nu, ny = HORIZON * DIM_U, HORIZON * DIM_Y
        self.g, self.u, self.y = cp.Variable(nc), cp.Variable(nu), cp.Variable(ny)
        self.uini = cp.Parameter(T_INI * DIM_U)
        self.eini = cp.Parameter(T_INI * DIM_E)
        self.yini = cp.Parameter(T_INI * DIM_Y)
        self.ef = cp.Parameter(HORIZON * DIM_E)
        self.r = cp.Parameter(ny)

        objective = 0.5 * cp.sum(cp.multiply(Q_BAR, cp.square(self.y - self.r)))
        objective += 0.5 * cp.sum(cp.multiply(R_BAR, cp.square(self.u)))
        objective += 0.5 * LAMBDA_CONSISTENCY * cp.sum_squares(blocks["Up"] @ self.g - self.uini)
        objective += 0.5 * LAMBDA_CONSISTENCY * cp.sum_squares(blocks["Ep"] @ self.g - self.eini)
        objective += LAMBDA_Y * cp.sum_squares(blocks["Yp"] @ self.g - self.yini)
        objective += 0.5 * LAMBDA_CONSISTENCY * cp.sum_squares(blocks["Uf"] @ self.g - self.u)
        objective += 0.5 * LAMBDA_CONSISTENCY * cp.sum_squares(blocks["Ef"] @ self.g - self.ef)
        objective += 0.5 * RHO_FUTURE_OUTPUT * cp.sum_squares(blocks["Yf"] @ self.g - self.y)
        objective += LAMBDA_G * cp.sum_squares(self.g)

        y_low = np.tile(np.r_[np.full(N_VEHICLES, VELOCITY_ERROR_MIN), np.full(N_CAV, SPACING_ERROR_MIN)], HORIZON)
        y_high = np.tile(np.r_[np.full(N_VEHICLES, VELOCITY_ERROR_MAX), np.full(N_CAV, SPACING_ERROR_MAX)], HORIZON)
        constraints = [self.u >= U_MIN, self.u <= U_MAX, self.y >= y_low, self.y <= y_high]
        self.problem = cp.Problem(cp.Minimize(objective), constraints)

    def solve(self, uini, eini, yini, ef, reference):
        _assign(self.uini, uini); _assign(self.eini, eini); _assign(self.yini, yini)
        _assign(self.ef, ef); _assign(self.r, reference)
        start = time.perf_counter()
        self.problem.solve(
        solver=cp.OSQP,
        warm_start=True,
        eps_abs=1e-4,
        eps_rel=1e-4,
        max_iter=20000,
        polishing=False,
        verbose=False,
        )
        elapsed = time.perf_counter() - start
        if self.problem.status not in (cp.OPTIMAL, cp.OPTIMAL_INACCURATE):
            raise RuntimeError(f"Soft DeeP-LCC status: {self.problem.status}")
        return {
            "u": np.asarray(self.u.value).reshape(HORIZON, DIM_U),
            "y": np.asarray(self.y.value).reshape(HORIZON, DIM_Y),
            "g": np.asarray(self.g.value),
            "objective": float(self.problem.value),
            "time": elapsed,
            "iterations": int(self.problem.solver_stats.num_iters or 0),
        }


class ReducedCDeepLCC:
    def __init__(self, blocks):
        self.B = blocks
        Up, Ep, Yp, Uf, Ef, Yf = (blocks[k] for k in ("Up", "Ep", "Yp", "Uf", "Ef", "Yf"))
        nc = Up.shape[1]
        nu, ny = HORIZON * DIM_U, HORIZON * DIM_Y

        A_fixed = fmat(np.vstack([Up, Ep, Yp, Ef]))
        A_decision = fmat(np.vstack([Uf, Yf]))
        fixed_weights = np.r_[
            np.full(Up.shape[0], LAMBDA_CONSISTENCY),
            np.full(Ep.shape[0], LAMBDA_CONSISTENCY),
            np.full(Yp.shape[0], 2.0 * LAMBDA_Y),
            np.full(Ef.shape[0], LAMBDA_CONSISTENCY),
        ]
        decision_weights = np.r_[
            np.full(nu, LAMBDA_CONSISTENCY),
            np.full(ny, RHO_FUTURE_OUTPUT),
        ]
        decision_cost = fmat(np.diag(np.r_[R_BAR, Q_BAR]))
        y_low_step = np.r_[np.full(N_VEHICLES, VELOCITY_ERROR_MIN), np.full(N_CAV, SPACING_ERROR_MIN)]
        y_high_step = np.r_[np.full(N_VEHICLES, VELOCITY_ERROR_MAX), np.full(N_CAV, SPACING_ERROR_MAX)]
        lower = np.r_[np.full(nu, U_MIN), np.tile(y_low_step, HORIZON)]
        upper = np.r_[np.full(nu, U_MAX), np.tile(y_high_step, HORIZON)]
        self.lower, self.upper = lower, upper
        self.dimension = len(lower)
        self.data = DeePCKForm()
        status = lib.deepc_kform_offline(
            ct.byref(self.data), ptr(A_fixed), ptr(A_decision),
            BINT(A_fixed.shape[0]), BINT(self.dimension), BINT(nc),
            ptr(fixed_weights), ptr(decision_weights), ct.c_double(2.0 * LAMBDA_G),
            ptr(decision_cost), ptr(lower), ptr(upper),
        )
        if status != 0:
            raise RuntimeError(f"deepc_kform_offline failed with status {status}")
        self.H = np.ctypeslib.as_array(self.data.Hphys, shape=(self.dimension ** 2,)).copy().reshape((self.dimension, self.dimension), order="F")
        self.F = np.ctypeslib.as_array(self.data.Fphys, shape=(self.dimension * self.data.nfixed,)).copy().reshape((self.dimension, self.data.nfixed), order="F")
        self.z_warm = None

    def fixed_data(self, uini, eini, yini, ef):
        return np.ascontiguousarray(np.r_[
            np.asarray(uini).reshape(-1), np.asarray(eini).reshape(-1),
            np.asarray(yini).reshape(-1), np.asarray(ef).reshape(-1),
        ], dtype=np.float64)

    def solve(self, uini, eini, yini, ef, reference, warm_start=True):
        start = time.perf_counter()
        fixed_data = self.fixed_data(uini, eini, yini, ef)
        tracking = np.ascontiguousarray(np.r_[
            np.zeros(HORIZON * DIM_U),
            -Q_BAR * np.asarray(reference).reshape(-1),
        ])
        z = np.zeros(self.dimension, dtype=np.float64)
        info = BoxQPInfo()
        warm = self.z_warm if warm_start else None
        status = lib.deepc_kform_online_step(
            ct.byref(self.data), ptr(fixed_data), ptr(tracking),
            None if warm is None else ptr(warm), ct.c_double(1e-7),
            BINT(60), ptr(z), ct.byref(info),
        )
        elapsed = time.perf_counter() - start
        if status not in (0, 1):
            raise RuntimeError(f"C BoxQP status={status}, chol_info={info.chol_info}")
        self.z_warm = np.ascontiguousarray(z.copy())
        g = self.recover_g(fixed_data, z)
        q = self.F @ fixed_data + tracking
        nu = HORIZON * DIM_U
        return {
            "u": z[:nu].reshape(HORIZON, DIM_U),
            "y": z[nu:].reshape(HORIZON, DIM_Y),
            "z": z,
            "g": g,
            "time": elapsed,
            "iterations": int(info.iterations),
            "mu": float(info.mu),
            "reduced_objective": float(0.5 * z @ self.H @ z + q @ z),
        }

    def recover_g(self, fixed_data, z):
        fixed_data = np.ascontiguousarray(fixed_data, dtype=np.float64)
        z = np.ascontiguousarray(z, dtype=np.float64)
        g = np.zeros(self.data.ncol, dtype=np.float64)
        status = lib.deepc_kform_recover_g(ct.byref(self.data), ptr(fixed_data), ptr(z), ptr(g))
        if status != 0:
            raise RuntimeError(f"deepc_kform_recover_g failed with status {status}")
        return g

    def full_objective(self, uini, eini, yini, ef, reference, u, y):
        B = self.B
        u0, e0, y0, ef0, r0 = [np.asarray(v).reshape(-1) for v in (uini, eini, yini, ef, reference)]
        uv, yv = np.asarray(u).reshape(-1), np.asarray(y).reshape(-1)
        fixed_data = np.ascontiguousarray(np.r_[u0, e0, y0, ef0])
        z = np.ascontiguousarray(np.r_[uv, yv])
        g = np.zeros(self.data.ncol, dtype=np.float64)
        status = lib.deepc_kform_recover_g(
            ct.byref(self.data), ptr(fixed_data), ptr(z), ptr(g)
        )
        if status != 0:
            raise RuntimeError(f"deepc_kform_recover_g failed with status {status}")
        value = 0.5 * np.sum(Q_BAR * (yv - r0)**2) + 0.5 * np.sum(R_BAR * uv**2)
        value += 0.5 * LAMBDA_CONSISTENCY * np.sum((B["Up"] @ g - u0)**2)
        value += 0.5 * LAMBDA_CONSISTENCY * np.sum((B["Ep"] @ g - e0)**2)
        value += LAMBDA_Y * np.sum((B["Yp"] @ g - y0)**2)
        value += 0.5 * LAMBDA_CONSISTENCY * np.sum((B["Uf"] @ g - uv)**2)
        value += 0.5 * LAMBDA_CONSISTENCY * np.sum((B["Ef"] @ g - ef0)**2)
        value += 0.5 * RHO_FUTURE_OUTPUT * np.sum((B["Yf"] @ g - yv)**2)
        value += LAMBDA_G * np.sum(g**2)
        return float(value)


class ReducedNativeOSQPDeepLCC:
    def __init__(self, c_controller):
        self.c = c_controller; self.H = c_controller.H; self.F = c_controller.F
        self.solver = osqp.OSQP()
        self.solver.setup(P=sp.csc_matrix(np.triu(self.H)), q=np.zeros(c_controller.dimension),
                          A=sp.eye(c_controller.dimension, format="csc"),
                          l=c_controller.lower, u=c_controller.upper, eps_abs=1e-7, eps_rel=1e-7,
                          max_iter=100000, verbose=False)

    def solve(self, uini, eini, yini, ef, reference):
        start = time.perf_counter()
        fixed = self.c.fixed_data(uini, eini, yini, ef)
        tracking = np.r_[np.zeros(HORIZON * DIM_U), -Q_BAR * np.asarray(reference).reshape(-1)]
        q = self.F @ fixed + tracking
        self.solver.update(q=q); answer = self.solver.solve(); elapsed = time.perf_counter() - start
        if answer.info.status_val not in (1, 2):
            raise RuntimeError(f"native OSQP failed: {answer.info.status}")
        z = np.asarray(answer.x).copy(); nu = HORIZON * DIM_U
        return {"u": z[:nu].reshape(HORIZON, DIM_U), "y": z[nu:].reshape(HORIZON, DIM_Y),
                "z": z, "g": self.c.recover_g(fixed, z), "time": elapsed,
                "iterations": int(answer.info.iter),
                "reduced_objective": float(0.5 * z @ self.H @ z + q @ z)}



## 6. Define one common closed-loop braking simulation


In [ ]:


# Replan at every simulation step for both controlled cases.
CONTROL_SHIFT = 1

# Emergency-braking layer for the original baseline only.
AEB_SAFETY_MARGIN = 0.5
AEB_TTC_LIMIT = 1.0


def apply_aeb_safety_filter(plant, action, v_head):
    action = np.asarray(action, dtype=np.float64).reshape(-1).copy()
    activations = 0
    for control_index, vehicle_index in enumerate(plant.pos_cav):
        lead_velocity = float(
            v_head if vehicle_index == 0 else plant.v[vehicle_index - 1]
        )
        follower_velocity = float(plant.v[vehicle_index])
        spacing = float(plant.s[vehicle_index])
        closing_speed = max(0.0, follower_velocity - lead_velocity)
        available_distance = max(
            spacing - (S_ST + AEB_SAFETY_MARGIN), 1.0e-6
        )
        required_deceleration = max(
            0.0,
            (follower_velocity**2 - lead_velocity**2)
            / (2.0 * available_distance),
        )
        time_to_bound = (
            available_distance / closing_speed
            if closing_speed > 1.0e-9
            else np.inf
        )
        if (
            required_deceleration >= abs(U_MIN)
            or time_to_bound <= AEB_TTC_LIMIT
            or spacing <= S_ST + AEB_SAFETY_MARGIN
        ):
            action[control_index] = U_MIN
            activations += 1
    return action, activations

# Keep False for the quick run. 
RUN_ORIGINAL_LCC = False


def run_closed_loop(
    mode,
    noise,
    head_velocity,
    show_progress=True,
):
    if mode not in {"all_hdv", "original", "fast_c"}:
        raise ValueError(
            "mode must be 'all_hdv', 'original', or 'fast_c'"
        )

    vehicle_id = (
        np.zeros(N_VEHICLES, dtype=int)
        if mode == "all_hdv"
        else ID
    )

    plant = MixedTrafficPlatoon(vehicle_id)

    # Each controller is constructed only once, before the online loop.
    hard_controller = (
        OriginalDeepLCC(BLOCKS)
        if mode == "original"
        else None
    )

    fast_controller = (
        ReducedCDeepLCC(BLOCKS)
        if mode == "fast_c"
        else None
    )

    u_window = np.zeros(
        (T_INI, DIM_U),
        dtype=np.float64,
    )
    e_window = np.zeros(
        (T_INI, DIM_E),
        dtype=np.float64,
    )
    y_window = np.zeros(
        (T_INI, DIM_Y),
        dtype=np.float64,
    )

    reference = np.zeros(
        HORIZON * DIM_Y,
        dtype=np.float64,
    )

    velocities = []
    spacings = []
    controls = []
    solve_times = []
    iterations = []
    aeb_activations = 0

    planned_actions = None
    number_of_solves = int(
        np.ceil(CONTROL_STEPS / CONTROL_SHIFT)
    )
    solve_number = 0

    for k in range(CONTROL_STEPS):
        if mode == "all_hdv":
            action = np.empty(0, dtype=np.float64)

        else:
            # Recompute the control plan only every CONTROL_SHIFT steps.
            if k % CONTROL_SHIFT == 0:
                solve_number += 1

                if show_progress:
                    print(
                        f"\r{mode}: solve "
                        f"{solve_number}/{number_of_solves}, "
                        f"simulation step {k}/{CONTROL_STEPS}",
                        end="",
                        flush=True,
                    )

                ef = (
                    head_velocity[k:k + HORIZON]
                    - V_STAR
                )

                if len(ef) < HORIZON:
                    ef = np.pad(
                        ef,
                        (0, HORIZON - len(ef)),
                        mode="edge",
                    )

                if mode == "original":
                    result = hard_controller.solve(
                        u_window,
                        e_window,
                        y_window,
                        ef,
                        reference,
                    )
                else:
                    result = fast_controller.solve(
                        u_window,
                        e_window,
                        y_window,
                        ef,
                        reference,
                    )

                available_actions = min(
                    CONTROL_SHIFT,
                    len(result["u"]),
                )

                planned_actions = (
                    result["u"][:available_actions].copy()
                )

                # If CONTROL_SHIFT exceeds the prediction horizon,
                # repeat the last planned action.
                if available_actions < CONTROL_SHIFT:
                    planned_actions = np.vstack([
                        planned_actions,
                        np.repeat(
                            planned_actions[-1:, :],
                            CONTROL_SHIFT - available_actions,
                            axis=0,
                        ),
                    ])

                solve_times.append(result["time"])
                iterations.append(result["iterations"])

            action_index = k % CONTROL_SHIFT
            action = planned_actions[action_index].copy()
            if mode == "original":
                action, triggered = apply_aeb_safety_filter(
                    plant, action, head_velocity[k]
                )
                aeb_activations += triggered

        measured = plant.step(
            action,
            head_velocity[k],
            noise[k],
        )

        velocities.append(plant.v.copy())
        spacings.append(plant.s.copy())

        if mode != "all_hdv":
            controls.append(action.copy())

            u_window = np.vstack([
                u_window[1:],
                action.reshape(1, -1),
            ])

            e_window = np.vstack([
                e_window[1:],
                [[head_velocity[k] - V_STAR]],
            ])

            y_window = np.vstack([
                y_window[1:],
                measured.reshape(1, -1),
            ])

    if show_progress and mode != "all_hdv":
        print()

    return {
        "velocity": np.asarray(
            velocities,
            dtype=np.float64,
        ),
        "spacing": np.asarray(
            spacings,
            dtype=np.float64,
        ),
        "control": np.asarray(
            controls,
            dtype=np.float64,
        ),
        "times": np.asarray(
            solve_times,
            dtype=np.float64,
        ),
        "iterations": np.asarray(
            iterations,
            dtype=np.float64,
        ),
        "aeb_activations": int(aeb_activations),
    }


## 7. Run the fixed original and C controller cases


In [ ]:
# Use one-step replanning with noise disabled for the paper benchmark.
CONTROL_SHIFT = 1
ONLINE_NOISE = 0.0
head_velocity, _ = brake_profile(CONTROL_STEPS)
shared_noise = np.zeros((CONTROL_STEPS, N_VEHICLES), dtype=np.float64)

print("Running original hard-equality DeeP-LCC...")
original_lcc = run_closed_loop("original", shared_noise, head_velocity)
print("Running C Fast-DeeP-LCC...")
fast_lcc = run_closed_loop("fast_c", shared_noise, head_velocity)

for name, result in [("Original DeeP-LCC", original_lcc), ("C Fast-DeeP-LCC", fast_lcc)]:
    print(f"{name}: mean online solve time = {1e3 * np.mean(result['times']):.3f} ms")
    print(f"{name}: AEB activations = {result['aeb_activations']}")

print("C Fast-DeeP-LCC minimum actual CAV spacing:",
      np.min(fast_lcc["spacing"][:, POS_CAV]), "m")


## 8. Plot CAV spacing with the shared AEB safety layer


In [ ]:
# CAV spacing comparison.

number_of_steps = min(
    len(original_lcc["spacing"]),
    len(fast_lcc["spacing"]),
)
time_s = np.arange(number_of_steps) * DT

with plt.rc_context({
    "font.size": 13,
    "axes.titlesize": 18,
    "axes.labelsize": 17,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 14,
}):
    fig, axes = plt.subplots(
        1,
        2,
        figsize=(11, 3.8),
        sharex=True,
        sharey=True,
        constrained_layout=True,
    )

    for axis, vehicle in zip(axes, POS_CAV):
        axis.plot(
            time_s,
            original_lcc["spacing"][:number_of_steps, vehicle],
            color="tab:blue",
            linewidth=2.4,
            label="Original DeeP-LCC",
        )

        axis.plot(
            time_s,
            fast_lcc["spacing"][:number_of_steps, vehicle],
            color="tab:orange",
            linestyle="--",
            linewidth=2.4,
            label="Fast-DeePC for LCC problem",
        )

        axis.axhline(
            5.0,
            color="red",
            linestyle=":",
            linewidth=1.6,
            label="spacing bounds",
        )
        axis.axhline(
            40.0,
            color="red",
            linestyle=":",
            linewidth=1.6,
        )

        axis.set_title(f"CAV {vehicle + 1} spacing")
        axis.set_xlabel("time (s)")
        axis.set_ylabel("spacing (m)")

        # Match the vertical scale in the reference figure.
        axis.set_ylim(3.0, 42.0)
        axis.set_yticks(np.arange(5.0, 40.1, 5.0))

        axis.set_xlim(time_s[0] - 0.8, time_s[-1] + 0.5)
        axis.grid(False)

    axes[0].legend(
        loc="center right",
        frameon=True,
    )

    spacing_plot_path = (
        ROOT / "media" / "mixed_traffic_cav_spacing.png"
    )

    fig.savefig(
        spacing_plot_path,
        dpi=250,
        bbox_inches="tight",
    )

    plt.show()

print("Saved spacing plot:", spacing_plot_path)

## 9. Write the mixed-traffic controller-comparison GIF


In [ ]:
# Animated mixed-traffic platoon comparison.Uses all_hdv, fast_lcc, and optionally original_lcc results.

import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.patches import Rectangle
from IPython.display import Image, display


FRAME_STRIDE = 2
GIF_FPS = 20
CAR_LENGTH = 4.5
CAR_HEIGHT = 1.15
VIEW_BEHIND_HEAD = 185.0
VIEW_AHEAD_HEAD = 18.0

gif_path = ROOT / "media" / "mixed_traffic.gif"


# Include every successfully completed simulation.
scenarios = [
    ("Original hard-equality DeeP-LCC", original_lcc),
    ("C Fast-DeeP-LCC BoxQP", fast_lcc),
]


def reconstruct_vehicle_positions(result):
    """
    Reconstruct longitudinal positions from vehicle spacing.

    Vehicle 0 is the external head vehicle. Vehicles 1-8 are the
    following mixed-traffic vehicles.
    """
    spacing = np.asarray(
        result["spacing"],
        dtype=np.float64,
    )

    number_of_steps = len(spacing)

    head_speed = np.asarray(
        head_velocity[:number_of_steps],
        dtype=np.float64,
    )

    # Integrate the head-vehicle velocity.
    head_position = np.cumsum(
        head_speed * DT
    )

    # Each spacing is the distance from one vehicle to its leader.
    following_positions = (
        head_position[:, None]
        - np.cumsum(spacing, axis=1)
    )

    return np.column_stack([
        head_position,
        following_positions,
    ])


# Precompute all positions.
for index, (name, result) in enumerate(scenarios):
    scenarios[index] = (
        name,
        result,
        reconstruct_vehicle_positions(result),
    )


maximum_steps = max(
    len(positions)
    for _, _, positions in scenarios
)

frame_steps = list(
    range(0, maximum_steps, FRAME_STRIDE)
)

if frame_steps[-1] != maximum_steps - 1:
    frame_steps.append(maximum_steps - 1)


# Vehicle colors.
HEAD_COLOR = "#202020"
HDV_COLOR = "#8c8c8c"
CAV_COLOR = "#f28e2b"
CAV_EDGE = "#9a4b00"
HDV_EDGE = "#404040"

number_of_panels = len(scenarios)

with plt.rc_context({
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
}):
    fig, axes = plt.subplots(
        number_of_panels,
        1,
        figsize=(12, 2.65 * number_of_panels),
        squeeze=False,
    )

    axes = axes[:, 0]
    animated_panels = []

    # Determine the complete world range for stationary road markings.
    largest_head_distance = max(
        positions[-1, 0]
        for _, _, positions in scenarios
    )

    world_minimum = -220.0
    world_maximum = largest_head_distance + 40.0

    for axis, (name, result, positions) in zip(
        axes,
        scenarios,
    ):
        axis.set_facecolor("#dceeff")

        # Road.
        axis.add_patch(
            Rectangle(
                (world_minimum, -2.35),
                world_maximum - world_minimum,
                4.70,
                facecolor="#4c4c4c",
                edgecolor="#202020",
                linewidth=1.2,
                zorder=0,
            )
        )

        # Road boundaries.
        axis.plot(
            [world_minimum, world_maximum],
            [-2.12, -2.12],
            color="white",
            linewidth=1.5,
            zorder=1,
        )
        axis.plot(
            [world_minimum, world_maximum],
            [2.12, 2.12],
            color="white",
            linewidth=1.5,
            zorder=1,
        )

        # Moving-world lane markings.
        for marker_position in np.arange(
            world_minimum,
            world_maximum,
            12.0,
        ):
            axis.plot(
                [
                    marker_position,
                    marker_position + 6.0,
                ],
                [0.0, 0.0],
                color="white",
                linewidth=1.3,
                alpha=0.75,
                zorder=1,
            )

        vehicle_patches = []
        vehicle_labels = []

        # Head vehicle plus eight following vehicles.
        for vehicle_index in range(N_VEHICLES + 1):
            if vehicle_index == 0:
                face_color = HEAD_COLOR
                edge_color = "black"
                label = "Head"
            else:
                platoon_index = vehicle_index - 1

                if ID[platoon_index] == 1:
                    face_color = CAV_COLOR
                    edge_color = CAV_EDGE
                    label = f"CAV {vehicle_index}"
                else:
                    face_color = HDV_COLOR
                    edge_color = HDV_EDGE
                    label = f"HDV {vehicle_index}"

            initial_x = positions[0, vehicle_index]

            vehicle_patch = Rectangle(
                (
                    initial_x - 0.5 * CAR_LENGTH,
                    -0.5 * CAR_HEIGHT,
                ),
                CAR_LENGTH,
                CAR_HEIGHT,
                facecolor=face_color,
                edgecolor=edge_color,
                linewidth=1.4,
                zorder=5,
            )

            axis.add_patch(vehicle_patch)
            vehicle_patches.append(vehicle_patch)

            vehicle_text = axis.text(
                initial_x,
                0.92,
                label,
                ha="center",
                va="bottom",
                fontsize=7.5,
                color="black",
                zorder=7,
            )

            vehicle_labels.append(vehicle_text)

        # Legend entries.
        axis.scatter(
            [],
            [],
            marker="s",
            s=85,
            color=HEAD_COLOR,
            edgecolor="black",
            label="Head vehicle",
        )
        axis.scatter(
            [],
            [],
            marker="s",
            s=85,
            color=HDV_COLOR,
            edgecolor=HDV_EDGE,
            label="HDV",
        )
        axis.scatter(
            [],
            [],
            marker="s",
            s=85,
            color=CAV_COLOR,
            edgecolor=CAV_EDGE,
            label="CAV",
        )

        information_text = axis.text(
            0.985,
            0.92,
            "",
            transform=axis.transAxes,
            ha="right",
            va="top",
            fontsize=9.5,
            color="black",
            bbox=dict(
                boxstyle="round,pad=0.28",
                facecolor="white",
                edgecolor="gray",
                alpha=0.92,
            ),
            zorder=10,
        )

        axis.set_title(name)
        axis.set_ylim(-2.8, 2.8)
        axis.set_yticks([])
        axis.set_ylabel("Road")
        axis.grid(False)

        initial_head = positions[0, 0]
        axis.set_xlim(
            initial_head - VIEW_BEHIND_HEAD,
            initial_head + VIEW_AHEAD_HEAD,
        )

        animated_panels.append({
            "result": result,
            "positions": positions,
            "patches": vehicle_patches,
            "labels": vehicle_labels,
            "text": information_text,
        })

    axes[-1].set_xlabel(
        "Longitudinal road position (m)"
    )

    axes[0].legend(
        loc="lower left",
        ncol=3,
        fontsize=9,
        framealpha=0.95,
    )

    fig.suptitle(
        "Mixed-traffic platoon during the head-vehicle braking disturbance",
        fontsize=15,
    )

    fig.subplots_adjust(
        left=0.07,
        right=0.985,
        bottom=0.09,
        top=0.91,
        hspace=0.33,
    )


    def update(frame_number):
        simulation_step = frame_steps[frame_number]
        artists = []

        for axis, panel in zip(
            axes,
            animated_panels,
        ):
            positions = panel["positions"]
            result = panel["result"]

            index = min(
                simulation_step,
                len(positions) - 1,
            )

            current_positions = positions[index]
            current_head_position = current_positions[0]

            # Scroll the viewing window with the head vehicle.
            axis.set_xlim(
                current_head_position - VIEW_BEHIND_HEAD,
                current_head_position + VIEW_AHEAD_HEAD,
            )

            for vehicle_index, (
                vehicle_patch,
                vehicle_label,
            ) in enumerate(zip(
                panel["patches"],
                panel["labels"],
            )):
                x_position = current_positions[vehicle_index]

                vehicle_patch.set_xy((
                    x_position - 0.5 * CAR_LENGTH,
                    -0.5 * CAR_HEIGHT,
                ))

                vehicle_label.set_position((
                    x_position,
                    0.92,
                ))

                artists.extend([
                    vehicle_patch,
                    vehicle_label,
                ])

            time_seconds = index * DT

            # Current minimum CAV spacing.
            current_spacing = np.asarray(
                result["spacing"][index],
                dtype=np.float64,
            )

            minimum_cav_spacing = np.min(
                current_spacing[POS_CAV]
            )

            panel["text"].set_text(
                f"time = {time_seconds:.2f} s\n"
                f"step = {index}\n"
                f"minimum CAV spacing = "
                f"{minimum_cav_spacing:.2f} m"
            )

            artists.append(panel["text"])

        return artists


    animation = FuncAnimation(
        fig,
        update,
        frames=len(frame_steps),
        interval=1000 / GIF_FPS,
        blit=False,
        repeat=True,
    )

    animation.save(
        gif_path,
        writer=PillowWriter(fps=GIF_FPS),
        dpi=120,
    )

    plt.close(fig)


print("Saved GIF:", gif_path)


## Same-problem timing, metrics, and data-length sweep
AEB is applied only to the original hard-$\ell_1$ DeeP-LCC trajectory above. The three same-problem quadratic methods below are unfiltered.

In [ ]:
def _lcc_timing(values):
    values = 1e3 * np.asarray(values, dtype=np.float64)
    return dict(mean_ms=values.mean(), std_ms=values.std(ddof=1) if len(values) > 1 else 0.0,
                p95_ms=np.percentile(values, 95), max_ms=values.max())


def run_same_problem_lcc(method, blocks=BLOCKS):
    plant = MixedTrafficPlatoon(ID)
    if method == "full": controller = SoftDeepLCC(blocks)
    elif method == "c": controller = ReducedCDeepLCC(blocks)
    elif method == "osqp":
        helper = ReducedCDeepLCC(blocks); controller = ReducedNativeOSQPDeepLCC(helper)
    else: raise ValueError(method)
    uw = np.zeros((T_INI, DIM_U)); ew = np.zeros((T_INI, DIM_E)); yw = np.zeros((T_INI, DIM_Y))
    reference = np.zeros(HORIZON * DIM_Y); planned = None
    outputs, controls, times, iterations, consistency, output_residual, objectives, violations = [], [], [], [], [], [], [], []
    aeb_count = 0
    for k in range(CONTROL_STEPS):
        if k % CONTROL_SHIFT == 0:
            ef = head_velocity[k:k + HORIZON] - V_STAR
            if len(ef) < HORIZON: ef = np.pad(ef, (0, HORIZON - len(ef)), mode="edge")
            result = controller.solve(uw, ew, yw, ef, reference)
            planned = result["u"][:CONTROL_SHIFT].copy(); times.append(result["time"]); iterations.append(result["iterations"])
            g, uplan, yplan = result["g"], result["u"].reshape(-1), result["y"].reshape(-1)
            behavioral = np.r_[blocks["Up"] @ g - uw.reshape(-1), blocks["Ep"] @ g - ew.reshape(-1),
                               blocks["Yp"] @ g - yw.reshape(-1), blocks["Uf"] @ g - uplan, blocks["Ef"] @ g - ef]
            consistency.append(np.linalg.norm(behavioral)); output_residual.append(np.linalg.norm(blocks["Yf"] @ g - yplan))
            if "reduced_objective" in result: objectives.append(result["reduced_objective"])
            else: objectives.append(np.nan)
        action = planned[k % CONTROL_SHIFT].copy()
        measured = plant.step(action, head_velocity[k], shared_noise[k])
        outputs.append(measured.copy()); controls.append(action.copy())
        violations.append(max(0.0, float(np.max(-plant.v)), float(np.max(plant.v - V_MAX)),
                              float(np.max(5.0 - plant.s[POS_CAV])), float(np.max(plant.s[POS_CAV] - 40.0))))
        uw = np.vstack([uw[1:], action]); ew = np.vstack([ew[1:], [[head_velocity[k] - V_STAR]]]); yw = np.vstack([yw[1:], measured])
    outputs, controls = np.asarray(outputs), np.asarray(controls)
    common_cost = float(np.sum(outputs ** 2 * Q_STEP) + np.sum(controls ** 2 * WEIGHT_U))
    return dict(outputs=outputs, controls=controls, times=np.asarray(times), iterations=np.asarray(iterations),
                consistency=np.asarray(consistency), output_residual=np.asarray(output_residual), objectives=np.asarray(objectives),
                tracking_rmse=float(np.sqrt(np.mean(outputs ** 2))), common_cost=common_cost,
                max_actual_violation=float(np.max(violations)), aeb_activations=int(aeb_count))


lcc_full = run_same_problem_lcc("full")
lcc_osqp = run_same_problem_lcc("osqp")
lcc_c = run_same_problem_lcc("c")
original_output = np.c_[original_lcc["velocity"] - V_STAR, original_lcc["spacing"][:, POS_CAV] - S_STAR]
original_violation = max(0.0, float(np.max(-original_lcc["velocity"])), float(np.max(original_lcc["velocity"] - V_MAX)),
                         float(np.max(5.0 - original_lcc["spacing"][:, POS_CAV])), float(np.max(original_lcc["spacing"][:, POS_CAV] - 40.0)))
original_lcc_metrics = {"tracking_rmse": float(np.sqrt(np.mean(original_output ** 2))),
    "common_cost": float(np.sum(original_output ** 2 * Q_STEP) + np.sum(original_lcc["control"] ** 2 * WEIGHT_U)),
    "max_actual_violation": float(original_violation)}
print("\nTIMING (wall-clock online time, ms)")
for name, result in [("Full soft quadratic / OSQP", lcc_full), ("Reduced BoxQP / native OSQP", lcc_osqp), ("Reduced BoxQP / C", lcc_c)]:
    print(name, _lcc_timing(result["times"]), "mean_iter=", float(np.mean(result["iterations"])), "max_iter=", int(np.max(result["iterations"])))
print("\nC PLANT METRICS:", {k: lcc_c[k] for k in ("tracking_rmse", "common_cost", "max_actual_violation")})
print("ORIGINAL HARD-L1 PLANT METRICS:", original_lcc_metrics)
print("Original hard-L1 AEB activations =", original_lcc["aeb_activations"])
print("Quadratic full/reduced/C AEB activations =", lcc_full["aeb_activations"], lcc_osqp["aeb_activations"], lcc_c["aeb_activations"])
print("C behavioral residuals: consistency mean/max =", lcc_c["consistency"].mean(), lcc_c["consistency"].max())
print("C behavioral residuals: future-output mean/max =", lcc_c["output_residual"].mean(), lcc_c["output_residual"].max())
print("Reduced OSQP/C trajectory RMSE =", np.sqrt(np.mean((lcc_osqp["outputs"] - lcc_c["outputs"]) ** 2)))
print("Reduced OSQP/C maximum control difference =", np.max(np.abs(lcc_osqp["controls"] - lcc_c["controls"])))
print("Reduced OSQP/C maximum objective difference =", np.nanmax(np.abs(lcc_osqp["objectives"] - lcc_c["objectives"])))


def lcc_timing_sweep(data_lengths=(300, 500, 800), repeats=1):
    rows = {"Full soft quadratic / OSQP": [], "Reduced BoxQP / native OSQP": [], "Reduced BoxQP / C": []}
    u0 = np.zeros(T_INI * DIM_U); e0 = np.zeros(T_INI * DIM_E); y0 = np.zeros(T_INI * DIM_Y); ef0 = np.zeros(HORIZON); ref = np.zeros(HORIZON * DIM_Y)
    for T in data_lengths:
        blocks = build_hankel_blocks(u_data[:T], e_data[:T], y_data[:T])
        full = SoftDeepLCC(blocks); c = ReducedCDeepLCC(blocks); native = ReducedNativeOSQPDeepLCC(c)
        full.solve(u0, e0, y0, ef0, ref); native.solve(u0, e0, y0, ef0, ref); c.solve(u0, e0, y0, ef0, ref)
        samples = {key: [] for key in rows}
        for _ in range(repeats):
            samples["Full soft quadratic / OSQP"].append(full.solve(u0, e0, y0, ef0, ref)["time"])
            samples["Reduced BoxQP / native OSQP"].append(native.solve(u0, e0, y0, ef0, ref)["time"])
            samples["Reduced BoxQP / C"].append(c.solve(u0, e0, y0, ef0, ref)["time"])
        for key in rows: rows[key].append(float(np.mean(samples[key])))
    return np.asarray(data_lengths), rows


In [ ]:
# Measure all three solvers versus offline data length T, then plot.

T_VALUES = [300, 500, 800, 1200]
NUMBER_OF_WINDOWS = 8

# Reconstruct the outputs from the saved Fast-DeePC run.

controls = np.asarray(fast_lcc["control"], dtype=np.float64)
velocities = np.asarray(fast_lcc["velocity"], dtype=np.float64)
spacings = np.asarray(fast_lcc["spacing"], dtype=np.float64)

# output(k) =[velocity errors of vehicles 1-8,spacing errors of CAVs 3 and 6]
outputs = np.hstack([
    velocities - V_STAR,
    spacings[:, POS_CAV] - S_STAR,
])

number_of_steps = min(
    CONTROL_STEPS,
    len(controls),
    len(outputs),
    len(head_velocity),
)

# Reconstruct the online windows seen by the controller.

u_window = np.zeros((T_INI, DIM_U), dtype=np.float64)
e_window = np.zeros((T_INI, DIM_E), dtype=np.float64)
y_window = np.zeros((T_INI, DIM_Y), dtype=np.float64)

reference = np.zeros(HORIZON * DIM_Y, dtype=np.float64)
all_problems = []

for k in range(number_of_steps):
    # Future head-vehicle velocity error.
    ef = head_velocity[k:k + HORIZON] - V_STAR

    if len(ef) < HORIZON:
        ef = np.pad(
            ef,
            (0, HORIZON - len(ef)),
            mode="edge",
        )

    # Store the optimization problem before applying action k.
    all_problems.append((
        u_window.reshape(-1).copy(),
        e_window.reshape(-1).copy(),
        y_window.reshape(-1).copy(),
        ef.reshape(-1).copy(),
        reference.copy(),
    ))

    # Add the realized action and output to the past windows.
    u_window = np.vstack([
        u_window[1:],
        controls[k].reshape(1, DIM_U),
    ])

    e_window = np.vstack([
        e_window[1:],
        np.array(
            [[head_velocity[k] - V_STAR]],
            dtype=np.float64,
        ),
    ])

    y_window = np.vstack([
        y_window[1:],
        outputs[k].reshape(1, DIM_Y),
    ])

# Choose representative problems across the experiment.
selected_indices = np.linspace(
    0,
    len(all_problems) - 1,
    min(NUMBER_OF_WINDOWS, len(all_problems)),
    dtype=int,
)

test_problems = [
    all_problems[index]
    for index in selected_indices
]

print("Selected simulation steps:", selected_indices)

timing_vs_T = {
    "Full soft quadratic / OSQP": [],
    "Reduced BoxQP / native OSQP": [],
    "Fast-DeePC / C BoxQP": [],
}

for T in T_VALUES:
    print(f"\nTesting T={T}...")

    blocks_T = build_hankel_blocks(
        u_data[:T],
        e_data[:T],
        y_data[:T],
    )

    # Controller construction is offline and is not timed here.
    full_solver = SoftDeepLCC(blocks_T)
    c_solver = ReducedCDeepLCC(blocks_T)
    native_solver = ReducedNativeOSQPDeepLCC(c_solver)

    # Untimed warm-up.
    warm_problem = test_problems[0]
    full_solver.solve(*warm_problem)
    native_solver.solve(*warm_problem)
    c_solver.solve(*warm_problem)

    full_times = []
    native_times = []
    c_times = []

    for problem in test_problems:
        full_result = full_solver.solve(*problem)
        native_result = native_solver.solve(*problem)
        c_result = c_solver.solve(*problem)

        full_times.append(float(full_result["time"]))
        native_times.append(float(native_result["time"]))
        c_times.append(float(c_result["time"]))

    full_mean = float(np.mean(full_times))
    native_mean = float(np.mean(native_times))
    c_mean = float(np.mean(c_times))

    timing_vs_T["Full soft quadratic / OSQP"].append(
        full_mean
    )
    timing_vs_T["Reduced BoxQP / native OSQP"].append(
        native_mean
    )
    timing_vs_T["Fast-DeePC / C BoxQP"].append(
        c_mean
    )

    print(f"  Full soft OSQP: {full_mean:.6f} s")
    print(f"  Reduced OSQP:   {native_mean:.6f} s")
    print(f"  C BoxQP:        {c_mean:.6f} s")

styles = {
    "Full soft quadratic / OSQP": "tab:blue",
    "Reduced BoxQP / native OSQP": "tab:green",
    "Fast-DeePC / C BoxQP": "tab:orange",
}

with plt.rc_context({
    "font.size": 12,
    "axes.labelsize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 11,
}):
    fig, ax = plt.subplots(figsize=(9, 5))

    for name, values in timing_vs_T.items():
        ax.plot(
            T_VALUES,
            values,
            marker="o",
            markersize=6,
            linewidth=2.5,
            color=styles[name],
            label=name,
        )

    ax.set_yscale("log")
    ax.set_ylim(1e-2, 1e2)
    ax.set_xlim(300, 1250)
    ax.set_xticks([400, 600, 800, 1000, 1200])

    ax.set_xlabel(r"offline data length $T$")
    ax.set_ylabel("mean online solve time (s)")

    ax.grid(
        True,
        which="major",
        color="gray",
        alpha=0.30,
        linewidth=0.8,
    )
    ax.grid(
        True,
        which="minor",
        color="gray",
        alpha=0.15,
        linewidth=0.5,
    )

    ax.legend(loc="upper left", frameon=True)
    fig.tight_layout()

    output_path = (
        ROOT
        / "media"
        / "mixed_traffic_timing_vs_T_measured.png"
    )
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, dpi=250, bbox_inches="tight")
    plt.show()

print("\nMeasured timing values:")
for name, values in timing_vs_T.items():
    print(name, values)

print("Saved:", output_path)

In [ ]:
def print_original_timing(name, times, iterations):
    times_ms = 1e3 * np.asarray(times, dtype=np.float64)
    iterations = np.asarray(iterations, dtype=np.float64)

    print(name)
    print(f"  mean time: {np.mean(times_ms):.3f} ms")
    print(f"  std time:  {np.std(times_ms, ddof=1):.3f} ms")
    print(f"  p95 time:  {np.percentile(times_ms, 95):.3f} ms")
    print(f"  max time:  {np.max(times_ms):.3f} ms")
    print(f"  mean iterations: {np.mean(iterations):.2f}")
    print(f"  max iterations:  {int(np.max(iterations))}")


print_original_timing(
    "Original hard-L1 DeeP-LCC",
    original_lcc["times"],
    original_lcc["iterations"],
)